# 캡스톤 템플릿 노트북 — 사전 등록 → 출처 → 분할·누출 → baseline → 평가 → 민감도 → 대조 → 보고

**등급**: 🟢 Mac 로컬 · Colab T4에서도 실행 확인(약 1.5분) · **예상 시간**: 약 1분 (데모 설정) · **다운로드**: 없음 (모듈 02의 ProteinGym 캐시와 ESM-2 8M 캐시)

이 노트북은 **프로젝트 골격**이다. 데모는 **트랙 A 전용**(ProteinGym 단일 치환)의 가장 작은 예로 채워져 있다. 질문은 "동결 ESM-2 8M 임베딩 probe가 위치 정보를 쓰는 단순 baseline보다, *새 위치*를 예측할 때 나은가?"다.

**다른 트랙에 쓸 때 바꿔야 하는 것** (✏️ 셀 설정만으로는 부족하다): `load_assay`(데이터 로딩·점검), `make_split`·`leak_stats`(분할 규칙과 그 누출 지표), 특징 만들기(§4), 지표 `rho`. 아래 인터페이스는 유지한다: `make_split(kind, seed) -> (tr, te, groups)`, `fit_predict(X, y, groups, tr, te) -> 예측`, 예측을 `PRED[(분할, 시드, 방법)] = (te, pred)`에 저장.

**이 골격이 보여 주는 것** (모듈 01~06에서 실제로 틀렸거나 틀리기 쉬웠던 것):
1. 설정(분할·지표·결정 규칙)을 첫 셀에서 **결과를 보기 전에** 고정하고, 결정 규칙은 **실행 가능한 함수**로 한 곳에 둔다.
2. 데이터 파일·모델 파일 해시, 모델 리비전, 패키지 버전, 코드 커밋(+작업 트리 변경 여부)을 기록한다 (모듈 05: 고정하지 않은 리비전이 섞인 적이 있다).
3. 분할마다 **train과의 겹침을 숫자로** 재고, test 그룹 수와 test 비율도 본다. 분할 이름을 믿지 않는다 (모듈 05).
4. baseline은 **파라미터를 학습하지 않는 것**(train 라벨만 쓰는 최근접 위치 평균) · 단순 특징 · 무작위 초기화를 포함하고 **같은 절차**로 평가한다 (모듈 04·05).
5. 비교는 **같은 분할에서 짝지어**, 불확실성은 **상관 단위의 그룹 부트스트랩이 가능할 때만** 계산한다. test 그룹이 너무 적으면 구간을 계산하지 않고 그렇게 쓴다 (이 데모가 그 경우다).
6. 한 번에 한 요인만 바꾸는 **민감도**와, 파이프라인을 점검하는 **음성 대조**를 넣는다. 음성 대조는 누출 부재의 *증명*이 아니라 진단이다 (모듈 05·06).
7. 설정은 test로 고르지 않는다. 사전에 고정했고 민감도는 보고만 한다 (모듈 04·06).

## 0. ✏️ 사전 등록 설정 (결과를 보기 전에 정한다)

In [ ]:
import numpy as np

RUN_NAME = "demo_dn7a_probe_vs_baselines"

# ✏️ 데이터·모델
ASSAY = "DN7A_SACS2_Tsuboyama_2023_1JIC"                       # ProteinGym 단일 치환 assay (모듈 02 캐시)
MODEL = ("facebook/esm2_t6_8M_UR50D", "c731040fcd8d73dceaa04b0a8e6329b345b0f5df")      # (저장소, 리비전)

# ✏️ 분할·반복
SPLITS = ["random", "position-block"]
BLOCK = 10                                                       # 위치 블록 크기. 블록은 분할의 단위이자 부트스트랩의 군집 단위
SEEDS = [0, 1, 2, 3, 4]                                          # 시드마다 분할과 무작위 초기화 모델이 바뀐다 (동결 사전학습 모델과 Ridge는 결정적)
BOOT = 500                                                       # 그룹 부트스트랩 재표본 수
MIN_TEST_GROUPS = 8                                              # 모든 시드의 test가 이 수 이상의 군집(블록)을 가져야 구간을 계산한다

# ✏️ 사전 지정한 주 비교와 **실행 가능한** 결정 규칙
PRIMARY = dict(split="position-block", method="probe", baseline="B2")
MIN_EFFECT = 0.02                                                # 실질적 의미가 있다고 보는 최소 Δ (Spearman)
MIN_WINS = 4                                                     # Δ > 0이어야 하는 시드 수 (전체 시드 중)
SENSITIVITY_BLOCKS = [3, 5, 10]                                  # 한 번에 한 요인: 블록 크기 (길이 55 단백질에서 블록이 5개 이상 남는 크기만 가능)


def decide(deltas, ci):
    """결정 규칙의 유일한 정의. deltas: 시드별 Δ(정의되지 않은 시드는 NaN), ci: (하한, 상한) 또는 None(계산 불가)."""
    deltas = np.asarray(deltas, float)
    deltas = deltas[np.isfinite(deltas)]                                           # Spearman이 정의되지 않은 시드는 제외하고 개수를 보고한다
    if len(deltas) < MIN_WINS:
        return "판정 불가(유효 시드 부족)", float("nan"), 0
    mean_d, wins = float(np.mean(deltas)), int(np.sum(deltas > 0))
    ok = mean_d >= MIN_EFFECT and wins >= MIN_WINS and (ci is None or ci[0] > 0)
    return ("지지" if ok else "지지되지 않음"), mean_d, wins


DECISION_RULE = (f"Δ(= 지표_method − 지표_baseline)의 시드 평균 ≥ {MIN_EFFECT}, Δ > 0인 시드 ≥ {MIN_WINS}/{len(SEEDS)}, "
                 f"그리고 구간을 계산할 수 있을 때(모든 시드의 test 군집 ≥ {MIN_TEST_GROUPS}) 95% 구간의 하한 > 0 → '지지'. 그 밖에는 '지지되지 않음'")
print("RUN:", RUN_NAME)
print("주 비교:", PRIMARY)
print("결정 규칙:", DECISION_RULE)

In [ ]:
import hashlib
import importlib.metadata as md
import json
import os
import subprocess
import time
import warnings
import zipfile
from pathlib import Path

import pandas as pd
from Bio.Align import substitution_matrices
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.metrics import make_scorer
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit, ShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")


def find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "SETUP.md").exists():
            return q
    return None


ROOT = find_root()
CACHE = (ROOT / ".cache") if ROOT is not None else Path("cache")
os.environ.setdefault("HF_HOME", str(CACHE / "hf"))
PG = CACHE / "proteingym"
OUT = CACHE / "capstone_runs" / RUN_NAME                         # git 제외 폴더: 실행 기록은 여기에만 쓴다
OUT.mkdir(parents=True, exist_ok=True)

import torch
from huggingface_hub import snapshot_download
from transformers import AutoConfig, AutoTokenizer, EsmModel
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE, "| 출력 폴더:", OUT)

## 1. 출처 기록 (provenance)
해시(**전체**)·리비전·버전·커밋과 **작업 트리 변경 여부**를 `provenance.json`에 남긴다. 정식 제출용 실행은 코드를 커밋한 뒤에 해서 `repo_dirty=False`로 남긴다. 라이선스는 **확인한 근거를 적고, 확인하지 못했으면 '미확인'**이라고 적는다(모듈 06 §7). 미확인 자료는 *공개·재배포 전에* 권한을 확인해야 한다.

In [ ]:
def sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while b := f.read(chunk):
            h.update(b)
    return h.hexdigest()


def git_info():
    try:
        commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=ROOT).stdout.strip()
        dirty = bool(subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True, cwd=ROOT).stdout.strip())
        return commit or "(git 정보 없음)", dirty
    except Exception:
        return "(git 정보 없음)", None


ZIP, REF = PG / "DMS_ProteinGym_substitutions.zip", PG / "DMS_substitutions.csv"
assert ZIP.exists() and REF.exists(), "모듈 02 노트북 01로 ProteinGym 파일을 먼저 받을 것"
mdir = Path(snapshot_download(MODEL[0], revision=MODEL[1], allow_patterns=["model.safetensors", "*.json", "vocab.txt"]))
commit, dirty = git_info()
PROV = dict(
    run=RUN_NAME, time=time.strftime("%Y-%m-%d %H:%M:%S"), repo_commit=commit, repo_dirty=dirty,
    data={"ProteinGym zip sha256": sha256(ZIP), "reference csv sha256": sha256(REF), "assay": ASSAY},
    model={"repo": MODEL[0], "revision": MODEL[1], "model.safetensors sha256": sha256(mdir / "model.safetensors")},
    packages={k: md.version(k) for k in ["numpy", "pandas", "scikit-learn", "scipy", "torch", "transformers"]},
    seeds=SEEDS, splits=SPLITS, block=BLOCK, boot=BOOT, min_test_groups=MIN_TEST_GROUPS, primary=PRIMARY, decision_rule=DECISION_RULE,
    licenses={"ESM-2 weights": "MIT (Hugging Face 메타데이터)", "ProteinGym 저장소 코드": "MIT (GitHub API)", "ProteinGym 개별 DMS 데이터": "미확인 (assay별 원 출처 확인 필요; 공개 전 확인)"},
)
(OUT / "provenance.json").write_text(json.dumps(PROV, ensure_ascii=False, indent=1))
print("provenance.json 저장 | 커밋", commit[:10], "| 작업 트리 변경 있음:", dirty)
print("데이터 sha256", PROV["data"]["ProteinGym zip sha256"][:16] + "… | 모델 파일 sha256", PROV["model"]["model.safetensors sha256"][:16] + "…")

## 2. 데이터 점검
불러온 직후에 **자체 점검(assert)**을 둔다: wt 잔기 일치, 서열 길이, 중복. 점검이 깨지면 이후 결과는 의미가 없다.

In [ ]:
ref = pd.read_csv(REF)


def load_assay(dms_id):
    row = ref[ref.DMS_id == dms_id].iloc[0]
    with zipfile.ZipFile(ZIP) as z:
        df = pd.read_csv(z.open(f"DMS_ProteinGym_substitutions/{row.DMS_filename}"))
    df = df[~df.mutant.str.contains(":")].copy()                                      # 단일 치환만
    df["wt"], df["pos"], df["mt"] = df.mutant.str[0], df.mutant.str[1:-1].astype(int), df.mutant.str[-1]
    assert all(row.target_seq[p - 1] == w for p, w in zip(df.pos, df.wt)), "wt 잔기가 target_seq와 다름"
    assert df.mutant.is_unique, "같은 변이가 중복되어 있다"
    assert df.mutated_sequence.str.len().nunique() == 1, "길이가 같지 않은 서열이 있다"
    return row.target_seq, df.reset_index(drop=True), row


WT, DF, ROW = load_assay(ASSAY)
Y = DF.DMS_score.values.astype(float)
BLOCKS = (DF.pos.values - 1) // BLOCK                                                  # 분할·부트스트랩의 군집 id (위치 블록)
print(f"{ASSAY}: 길이 {len(WT)} | 단일 치환 {len(DF)}개 | 위치 {DF.pos.nunique()}개 | 블록({BLOCK}) {len(np.unique(BLOCKS))}개 | 유형 {ROW.coarse_selection_type}/{ROW.selection_type}")
print(f"점수 분포: 중앙값 {np.median(Y):.2f}, 5/95% [{np.percentile(Y, 5):.2f}, {np.percentile(Y, 95):.2f}] | 결측 {int(np.isnan(Y).sum())}")

## 3. 분할과 누출 측정
분할이 막으려는 것을 먼저 적는다: **random**은 아무것도 막지 않고, **position-block**은 "같은 위치의 다른 치환이 train에 있는" 경로를 막는다. 이름을 믿지 말고 숫자로 확인한다. test 변이 중 (a) 위치가 train에도 있는 비율, (b) 같은 (wt→mt) 쌍이 train에 있는 비율, (c) train의 가장 가까운 위치까지의 거리 중앙값을 본다. 그리고 **test 비율과 test 군집(블록) 수**를 본다. 이 수가 불확실성을 계산할 수 있는지를 정한다. (블록 크기가 균일하지 않으면 test 비율이 20%에서 벗어난다. 여기서는 마지막 블록이 5개 위치다.)

In [ ]:
def make_split(kind, seed, block=BLOCK):
    if kind == "random":
        tr, te = next(ShuffleSplit(n_splits=1, test_size=0.2, random_state=seed).split(DF))
        groups = np.arange(len(DF))                                                   # 학습 안 CV의 그룹: 변이 단위
    else:
        groups = (DF.pos.values - 1) // block
        assert len(np.unique(groups)) >= 5, f"블록 크기 {block}이 너무 커서 그룹이 {len(np.unique(groups))}개뿐이다 — 분할·내부 CV가 성립하지 않는다"
        tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed).split(DF, groups=groups))
    return tr, te, groups


def leak_stats(tr, te):
    pos_tr = np.sort(DF.pos.values[tr])
    seen_pos = np.isin(DF.pos.values[te], pos_tr).mean()
    pair_tr = set(zip(DF.wt.values[tr], DF.mt.values[tr]))
    seen_pair = np.mean([(w, m) in pair_tr for w, m in zip(DF.wt.values[te], DF.mt.values[te])])
    dist = np.min(np.abs(DF.pos.values[te][:, None] - pos_tr[None, :]), axis=1)
    return dict(test_비율=len(te) / len(DF), test_위치가_train에_있음=seen_pos, 같은_wt_mt쌍이_train에_있음=seen_pair, train까지_위치거리_중앙값=float(np.median(dist)), test_군집수=len(np.unique(BLOCKS[te])))


rows = []
for kind in SPLITS:
    st = pd.DataFrame([leak_stats(*make_split(kind, s)[:2]) for s in SEEDS])
    rows.append(dict(분할=kind, **st.mean().round(3).to_dict(), 최소_test_군집수=int(st["test_군집수"].min())))
LEAK = pd.DataFrame(rows)
print(LEAK.to_string(index=False))
CI_OK = {kind: int(LEAK.set_index("분할").loc[kind, "최소_test_군집수"]) >= MIN_TEST_GROUPS for kind in SPLITS}
print(f"\n그룹 부트스트랩 구간을 계산할 수 있는가 (모든 시드의 test 군집 ≥ {MIN_TEST_GROUPS}):", CI_OK)

## 4. 특징: baseline과 모델
- **B0 (파라미터를 학습하지 않음)**: 각 test 변이를 *train에서 가장 가까운 위치*의 평균 DMS 점수로 예측(동률이면 양쪽 평균). train 라벨은 쓰지만 test 정보는 쓰지 않는다. random 분할에서는 같은 위치의 평균이 된다.
- **B1**: wt·mutant 원-핫 + BLOSUM62 (위치 정보 없음), **B2**: B1 + 위치 원-핫 (같은 위치가 train에 있으면 유리).
- **probe**: 동결 ESM-2 8M, 변이 위치의 마지막 층 은닉 상태. **probe-randinit**: 같은 구조의 *무작위 초기화* 모델 — **시드마다 새로 만든다**(시드 s로 초기화).
- B1, B2, probe, probe-randinit는 **같은 Ridge 파이프라인**(표준화 → Ridge, 규제 세기는 학습 안 그룹 CV로 Spearman 기준 선택)을 쓴다.

In [ ]:
AA = "ACDEFGHIKLMNPQRSTVWY"
BLOSUM = substitution_matrices.load("BLOSUM62")


def onehot(chars):
    M = np.zeros((len(chars), len(AA)))
    M[np.arange(len(chars)), [AA.index(c) for c in chars]] = 1
    return M


blos = np.array([BLOSUM[w][m] for w, m in zip(DF.wt, DF.mt)], dtype=float)[:, None]
B1 = np.hstack([onehot(DF.wt), onehot(DF.mt), blos])
P = np.zeros((len(DF), len(WT)))
P[np.arange(len(DF)), DF.pos.values - 1] = 1
B2 = np.hstack([B1, P])

tok = AutoTokenizer.from_pretrained(mdir)
ids_all = tok(DF.mutated_sequence.tolist(), return_tensors="pt")["input_ids"]
pos_all = torch.tensor(DF.pos.values)


@torch.no_grad()
def site_features(model, batch=128):
    model = model.eval().to(DEVICE)
    out = []
    for s in range(0, len(ids_all), batch):
        ids = ids_all[s:s + batch].to(DEVICE)
        h = model(input_ids=ids, attention_mask=torch.ones_like(ids)).last_hidden_state
        out.append(h[torch.arange(len(ids)), pos_all[s:s + batch].to(DEVICE)].float().cpu())
    return torch.cat(out).numpy()


def predict_nearest_position_mean(tr, te):
    pos_tr = np.unique(DF.pos.values[tr])
    mean_by_pos = {p: Y[tr][DF.pos.values[tr] == p].mean() for p in pos_tr}
    out = []
    for p in DF.pos.values[te]:
        d = np.abs(pos_tr - p)
        out.append(np.mean([mean_by_pos[q] for q in pos_tr[d == d.min()]]))            # 동률(양쪽 같은 거리)이면 평균
    return np.array(out)


t0 = time.time()
X_probe = site_features(EsmModel.from_pretrained(mdir, add_pooling_layer=False))
X_RAND = {}
for s in SEEDS:
    torch.manual_seed(s)
    X_RAND[s] = site_features(EsmModel(AutoConfig.from_pretrained(mdir), add_pooling_layer=False))        # 시드마다 다른 무작위 초기화
print(f"임베딩 {time.time() - t0:.0f}초 | probe {X_probe.shape} | 무작위 초기화 {len(X_RAND)}개 모델")
STATIC_FEATS = {"B1": B1, "B2": B2, "probe": X_probe}

## 5. 평가
시드마다 (분할 → 모든 방법)을 **같은 train/test**로 평가한다. 예측을 저장해 두고 짝지은 차이와 부트스트랩에 재사용한다. 음성 대조 **probe-shuffled**는 학습 라벨을 섞어 학습한다. **시드 평균이 0 근처**이고 부호가 일정하지 않아야 한다. 이것은 파이프라인 *진단*이지 누출 부재의 증명이 아니다(모든 누출 경로를 검출하지 못한다). 이상하면 원인을 조사하고 보고한다. test가 소수의 위치에 몰려 있으면(test 그룹 수가 유효 표본 크기의 대리값) 시드 하나의 우연 변동은 ±0.3까지도 커진다.

Spearman이 정의되지 않는 경우(상수 예측)는 **NaN으로 기록**하고 개수를 센다. (학습 안 CV로 규제 세기를 고를 때만 -1로 바꿔 선택에서 제외한다.)

In [ ]:
def rho(y, p):
    """평가용: 정의되지 않으면 NaN."""
    return float(spearmanr(y, p).statistic)


def rho_cv(y, p):
    """규제 세기 선택(학습 안 CV) 전용: 정의되지 않으면 -1로 두어 그 α를 고르지 않게 한다."""
    r = spearmanr(y, p).statistic
    return -1.0 if np.isnan(r) else float(r)


def fit_predict(X, y, groups, tr, te):
    pipe = make_pipeline(StandardScaler(), Ridge())
    n_cv = min(3, len(np.unique(groups[tr])))                                      # 학습 안 그룹 수가 3보다 작으면 접힘 수를 줄인다
    gs = GridSearchCV(pipe, {"ridge__alpha": np.logspace(-1, 5, 13)}, scoring=make_scorer(rho_cv), cv=GroupKFold(n_splits=n_cv).split(X[tr], y[tr], groups[tr]))
    gs.fit(X[tr], y[tr])
    return gs.predict(X[te])


PRED = {}                                                       # (분할, 시드, 방법) -> (te, pred)
rows = []
t0 = time.time()
for kind in SPLITS:
    for seed in SEEDS:
        tr, te, groups = make_split(kind, seed)
        PRED[(kind, seed, "B0")] = (te, predict_nearest_position_mean(tr, te))
        for name, X in {**STATIC_FEATS, "probe-randinit": X_RAND[seed]}.items():
            PRED[(kind, seed, name)] = (te, fit_predict(X, Y, groups, tr, te))
        y_shuf = Y.copy()
        y_shuf[tr] = np.random.default_rng(seed).permutation(Y[tr])
        PRED[(kind, seed, "probe-shuffled")] = (te, fit_predict(X_probe, y_shuf, groups, tr, te))
        for name in ["B0", "B1", "B2", "probe", "probe-randinit", "probe-shuffled"]:
            rows.append(dict(split=kind, seed=seed, method=name, rho=rho(Y[PRED[(kind, seed, name)][0]], PRED[(kind, seed, name)][1])))
RES = pd.DataFrame(rows)
print(f"평가 {time.time() - t0:.0f}초 | 정의되지 않은 Spearman(NaN) {int(RES.rho.isna().sum())}개")
print(RES.groupby(["split", "method"]).rho.agg(["mean", "std"]).round(3).unstack(0).to_string())

### 주 비교: 짝지은 차이, 시드별 값, 그리고 (가능할 때만) 그룹 부트스트랩
사전 지정한 주 비교의 Δ = probe − B2를 시드별로 구한다(같은 train/test). **시드는 같은 1,008개 변이에서 분할을 다시 뽑은 것이라 test 집합이 서로 겹친다. 독립 반복이 아니라 분할 민감도를 보는 것**이다. 구간은 각 시드의 test에서 **위치 블록을 재표본**한 뒤 시드에 걸쳐 평균해 만든다(조건부 구간: 이미 계산한 예측에 대한 구간이며 모델 재학습·분할 변동은 담지 않고, 시드 간 중첩도 반영하지 못한다). **모든 시드의 test 군집이 `MIN_TEST_GROUPS` 미만이면 구간을 계산하지 않는다.** 군집 2개를 복원추출하면 가능한 재표본이 3가지뿐이라 구간이 무의미하다.

In [ ]:
def seed_deltas(kind, m1, m2):
    return np.array([rho(Y[PRED[(kind, s, m1)][0]], PRED[(kind, s, m1)][1]) - rho(Y[PRED[(kind, s, m2)][0]], PRED[(kind, s, m2)][1]) for s in SEEDS])


def boot_delta(kind, m1, m2, B=BOOT, seed=123):
    """위치 블록 그룹 부트스트랩. 모든 시드의 test 군집 수가 MIN_TEST_GROUPS 이상일 때만 (하한, 상한)을, 아니면 None."""
    per_seed = []
    for s in SEEDS:
        te, p1 = PRED[(kind, s, m1)]
        _, p2 = PRED[(kind, s, m2)]
        per_seed.append((te, p1, p2, BLOCKS[te]))
    if min(len(np.unique(g)) for *_, g in per_seed) < MIN_TEST_GROUPS:
        return None
    rng = np.random.default_rng(seed)
    draws = np.zeros(B)
    for b in range(B):
        ds = []
        for te, p1, p2, g in per_seed:
            u = np.unique(g)
            idx = np.concatenate([np.where(g == k)[0] for k in rng.choice(u, size=len(u), replace=True)])
            ds.append(rho(Y[te][idx], p1[idx]) - rho(Y[te][idx], p2[idx]))
        draws[b] = np.nanmean(ds)
    return tuple(np.percentile(draws, [2.5, 97.5]))


rows = []
for kind in SPLITS:
    for m1, m2 in [("probe", "B2"), ("probe", "B1"), ("probe", "B0"), ("probe", "probe-randinit")]:
        d = seed_deltas(kind, m1, m2)
        ci = boot_delta(kind, m1, m2)
        ok_ = np.isfinite(d)
        rows.append({"분할": kind, "비교": f"{m1} − {m2}", "Δ 시드 평균": round(float(np.nanmean(d)), 3), "시드별 Δ": ", ".join(f"{x:+.2f}" for x in d),
                     "이긴 시드": f"{int((d[ok_] > 0).sum())}/{int(ok_.sum())}(유효)", "구간": "계산 안 함(test 군집 부족)" if ci is None else f"[{ci[0]:+.3f}, {ci[1]:+.3f}]"})
CMP = pd.DataFrame(rows)
print(CMP.to_string(index=False))

d = seed_deltas(PRIMARY["split"], PRIMARY["method"], PRIMARY["baseline"])
ci = boot_delta(PRIMARY["split"], PRIMARY["method"], PRIMARY["baseline"])
verdict, mean_d, wins = decide(d, ci)
print(f"\n[사전 지정 결정] {PRIMARY}\n  Δ 시드 평균 {mean_d:+.3f} (기준 ≥ {MIN_EFFECT}), 이긴 시드 {wins}/{len(SEEDS)} (기준 ≥ {MIN_WINS}), 구간 {'계산 안 함' if ci is None else ci} → '{verdict}'")

### 부트스트랩 기계의 자체 점검 (합성 데이터)
이 데모에서는 test 군집이 적어 구간을 계산하지 않았으므로, 구간 코드가 **충분한 군집이 있을 때 맞게 동작하는지**를 합성 데이터로 점검한다: 40개 군집 × 군집당 10개, 예측 p1·p2가 정답 y에 같은 정확도(Δ 참값 0)이면 구간이 0을 포함하고, p1이 훨씬 정확하면(Δ > 0) 포함하지 않아야 한다.

In [ ]:
def synthetic_ci(noise1, noise2, n_groups=40, per=10, B=400, seed=0):
    rng = np.random.default_rng(seed)
    g = np.repeat(np.arange(n_groups), per)
    y = rng.normal(size=n_groups)[g] + rng.normal(size=n_groups * per) * 0.5                 # 군집 효과 + 잡음
    p1, p2 = y + rng.normal(size=len(y)) * noise1, y + rng.normal(size=len(y)) * noise2
    draws = []
    for _ in range(B):
        idx = np.concatenate([np.where(g == k)[0] for k in rng.choice(n_groups, size=n_groups, replace=True)])
        draws.append(rho(y[idx], p1[idx]) - rho(y[idx], p2[idx]))
    return float(rho(y, p1) - rho(y, p2)), np.percentile(draws, [2.5, 97.5])


for label, (n1, n2) in {"같은 정확도(Δ 참값 0)": (1.0, 1.0), "p1이 훨씬 정확(Δ > 0)": (0.3, 1.5)}.items():
    pt, (lo, hi) = synthetic_ci(n1, n2)
    print(f"{label}: Δ = {pt:+.3f}, 구간 [{lo:+.3f}, {hi:+.3f}] → 0 {'포함' if lo <= 0 <= hi else '미포함'}")

## 6. 민감도: 한 번에 한 요인 (블록 크기)
주 비교가 블록 크기에 좌우되는가? **설정을 이 표로 고르지 않는다.** 결론이 사전 지정한 설정에 얼마나 의존하는지 보고하는 용도다. 블록 크기를 바꾸면 분할이 달라지므로 baseline도 같은 분할에서 다시 평가한다. 블록이 작을수록 test 군집은 늘지만 인접 위치가 train에 남는다(누출 방지가 약해진다).

In [ ]:
rows = []
t0 = time.time()
for blk in SENSITIVITY_BLOCKS:
    store, ng = {}, []
    for seed in SEEDS:
        tr, te, groups = make_split("position-block", seed, block=blk)
        ng.append(len(np.unique(groups[te])))
        for name in ["B2", "probe"]:
            store[(seed, name)] = (te, fit_predict(STATIC_FEATS[name], Y, groups, tr, te))
    d = np.array([rho(Y[store[(s, "probe")][0]], store[(s, "probe")][1]) - rho(Y[store[(s, "B2")][0]], store[(s, "B2")][1]) for s in SEEDS])
    rows.append({"블록 크기": blk, "test 군집 수(최소)": min(ng), "Δ(probe−B2) 시드 평균": d.mean(), "시드별": ", ".join(f"{x:+.2f}" for x in d), "이긴 시드": f"{int((d > 0).sum())}/{len(SEEDS)}"})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print(f"민감도 {time.time() - t0:.0f}초")

## 7. 결과 저장과 보고 문장 점검
표를 파일로 저장하고, 보고서에 붙일 **숫자 문장**을 코드에서 만든다. 서술의 숫자를 손으로 옮기다 틀리는 일(모듈 04·05)을 줄이기 위해서다.

In [ ]:
(OUT / "results_by_method.csv").write_text(RES.to_csv(index=False))
(OUT / "comparisons.txt").write_text(CMP.to_string(index=False))
print("저장:", sorted(p.name for p in OUT.iterdir()))
m = RES.groupby(["split", "method"]).rho.mean()
print("\n[자동 숫자 문장]")
for kind in SPLITS:
    print(f"- {kind}: " + ", ".join(f"{k} {m[(kind, k)]:.2f}" for k in ["B0", "B1", "B2", "probe", "probe-randinit", "probe-shuffled"]))
sh = RES[RES.method == "probe-shuffled"].groupby("split").rho.agg(["mean", "std", lambda v: v.abs().max()])
for kind in SPLITS:
    print(f"- 음성 대조(라벨 섞음) {kind}: 시드 평균 {sh.loc[kind].iloc[0]:+.2f}, SD {sh.loc[kind].iloc[1]:.2f}, |ρ| 최대 {sh.loc[kind].iloc[2]:.2f} (진단: 평균이 0에서 멀면 조사한다; test 군집이 적으면 변동이 크다)")

## 8. ✏️ 한계와 미실시 목록 (직접 채운다)
보고서에는 **하지 않은 것**을 적는다. 이 데모에서 예시:
- 한 assay, 한 모델 크기(8M), 시드 5개(test 집합이 겹치는 분할 반복), 규제 외 하이퍼파라미터 없음(Ridge만), 마지막 층만.
- test 군집(블록)이 2~6개뿐이라 그룹 부트스트랩 구간을 계산하지 않았다. 결정은 시드별 Δ와 효과 크기 기준으로 했다.
- 무작위 초기화 대조군은 시드 5개(서로 다른 초기화), 사전학습 모델은 하나.
- 데이터 라이선스 중 assay별 원 출처는 확인하지 못했다.
- 결과를 다른 assay·모델로 일반화하지 않는다.